# Data Cleaning Pipeline

In [32]:
import pandas as pd
from fuzzywuzzy import process

c:\Users\RhysL\Desktop\DE_Tools\venv\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [ ]:


# Sample DataFrame
data = {
    'User ID': [' 12345 ', '67890', ' 11223 ', '44556', '77889 '],   
    'Full Name': ['John Doe ', 'alice SMITH', 'Bob Johnson', 'Eve Adams ', 'CHRIS EVANS '],  
    'Date of Birth': ['2025-07-10', '1995-03-25', '2000-11-15', '2024-01-01', '2010-09-30'],  
    'Email': ['USER@GMAIL.COM ', 'alice.smith@yahoo.com', 'invalid-email.com', ' eve@outlook.com ', 'chris_evans@EMAIL.com'],  
    'Age': [' Twenty ', '30', '45', '18', ' Unknown '],  
    'Subscription': ['monthly', 'yearly', 'MONTHLY', 'liftime', 'monthyl'],  
    'Measurement': ['1.6bnp4hp', '2.5kw', '3.7', '1.5L', '4.3a8b'],  
    'Phone Number': ['123-456-7890', '9876543210', '(555) 123-4567', '444-5555', '999.888.7777']  
}

df= pd.DataFrame(data)

# Convert to DataFrame and transpose it
df_long = pd.DataFrame(data).T  # Transpose

# Reset index so the column names remain in the first row
df_long.reset_index(inplace=True)
# Rename columns
# print(df_long)
# features=['User ID', 'Full Name', 'Date of Birth', 'Email', 'Age', 'Phone Number']
# print(df[features])
print(df)
df_raw=df


## Phase 1: Broad Cleaning (Applied to All Columns)

### Functions

1. Standardize Column Names  
   - Convert to lowercase, remove spaces, replace special characters.  

In [4]:
def standardize_column_names(df):
    """Convert column names to lowercase, replace spaces with underscores."""
    df.columns = (
        df.columns.str.strip()       # Remove leading/trailing spaces
                .str.lower()         # Convert to lowercase
                .str.replace(' ', '_', regex=True) # Replace spaces with underscores
    )
    return df

2. Fix Data Types  #TODO!
   - Convert numeric-looking text columns to numbers.  
   - Convert date columns to proper datetime format.  
   - Ensure boolean values are consistent.  

3. Trim Whitespace & Standardize Case  
   - Remove leading/trailing spaces.  
   - Convert text to lowercase for consistency.  

In [5]:
def trim_whitespace(df):
    """Trim whitespace from all string columns."""
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    return df

### Testing

In [8]:
# Before
df.head()

,User ID,Full Name,Date of Birth,Email,Age,Subscription,Measurement,Phone Number
0,12345,John Doe,2025-07-10,USER@GMAIL.COM,Twenty,monthly,1.6bnp4hp,123-456-7890
1,67890,alice SMITH,1995-03-25,alice.smith@yahoo.com,30,yearly,2.5kw,9876543210
2,11223,Bob Johnson,2000-11-15,invalid-email.com,45,MONTHLY,3.7,(555) 123-4567
3,44556,Eve Adams,2024-01-01,eve@outlook.com,18,lifetime,1.5L,444-5555
4,77889,CHRIS EVANS,2010-09-30,chris_evans@EMAIL.com,Unknown,monthyl,4.3a8b,999.888.7777


In [20]:
# Apply Phase 1
df = standardize_column_names(df)
df.head()

,user_id,full_name,date_of_birth,email,age,subscription,measurement,phone_number
0,12345,John Doe,2025-07-10,USER@GMAIL.COM,Twenty,monthly,1.6bnp4hp,123-456-7890
1,67890,alice SMITH,1995-03-25,alice.smith@yahoo.com,30,yearly,2.5kw,9876543210
2,11223,Bob Johnson,2000-11-15,invalid-email.com,45,MONTHLY,3.7,(555) 123-4567
3,44556,Eve Adams,2024-01-01,eve@outlook.com,18,liftime,1.5L,444-5555
4,77889,CHRIS EVANS,2010-09-30,chris_evans@EMAIL.com,Unknown,monthyl,4.3a8b,999.888.7777


In [ ]:
df[['user_id','date_of_birth']].iloc[0]

user_id               12345
date_of_birth    2025-07-10
Name: 0, dtype: object

In [ ]:
df = trim_whitespace(df)
df.head()

## Phase 2: Targeted Cleaning (Specific to Data Type or Context)

In [35]:
df=df_raw

### Functions

Dictionary for renaming columns

In [36]:
rename_dict = {
    'User ID': 'user_id',
    'Full Name': 'full_name',
    'Subscription': 'subscription'
}

# Apply renaming
df.rename(columns=rename_dict, inplace=True)

df.columns

Index(['user_id', 'full_name', 'Date of Birth', 'Email', 'Age', 'subscription',
       'Measurement', 'Phone Number'],
      dtype='object')

Clean Categorical Variables  
   - Standardize known categories using a predefined dictionary.  


In [37]:
# Dictionary for standardizing subscription values
subscription_dict = {
    'monthly': 'Monthly',
    'MONTHLY': 'Monthly',
    'yearly': 'Yearly',
    'YEARLY': 'Yearly'
}

# Apply mapping to the 'subscription_plan' column
df['subscription'] = df['subscription'].replace(subscription_dict)

print(df['subscription'])


0    Monthly
1     Yearly
2    Monthly
3    liftime
4    monthyl
Name: subscription, dtype: object


   - Correct spelling errors using predefined mapping and fuzzy matching for unknown errors.  

In [38]:
# Generalized fuzzy matching function
def fuzzy_correct(value, valid_values, threshold=80):
    """
    Corrects a given value by finding the closest match in valid_values using fuzzy matching.
    
    Parameters:
    - value: The value to correct
    - valid_values: List of known valid values
    - threshold: Minimum confidence score required for replacement

    Returns:
    - Best match if confidence is above the threshold, otherwise returns the original value.
    """
    best_match, score = process.extractOne(value, valid_values)
    return best_match if score >= threshold else value  # Only replace if confidence is high


In [39]:
# List of valid categories
valid_subscriptions = ['Monthly', 'Yearly', 'Lifetime']
# Apply fuzzy matching
df['subscription'] = df['subscription'].apply(lambda x: fuzzy_correct(x, valid_subscriptions))

In [41]:
df[['user_id','subscription']].head()

,user_id,subscription
0,12345,Monthly
1,67890,Yearly
2,11223,Monthly
3,44556,Lifetime
4,77889,Monthly


Clean Numeric Variables


7. Extract Numeric Values from Measurement Columns  
   - Extract numeric values from mixed-format columns (e.g., `"1.5kw"` → `1.5`).  
   - Flag non-standard measurement formats containing multiple non-numeric segments (e.g., `"1.6bnp4hp"`).  



5. Validate Specific Fields  
   - Ensure emails, phone numbers, and other structured fields follow proper formats.  
   - Remove future birthdates and invalid date values.  



6. Apply Business Rules  
   - Example: Assign an "age_group" based on an age threshold.  


### Testing

## Phase 3: Final Checks & Quality Assurance

8. Re-check Missing Values  
   - Identify remaining gaps and decide on appropriate handling (e.g., imputation, removal).  


9. Summary Statistics  
   - Generate descriptive stats to validate the dataset's integrity.  
